IMPORTAMOS EL ARCHIVO

In [ ]:
import pandas as pd
import glob
import datetime as dt
import os
from dotenv import load_dotenv

# Cargar variables desde el archivo .env
load_dotenv()

# Obtener la zona horaria (y usar 'UTC' por defecto si no existe la variable)
user_tz = os.getenv('USER_TIMEZONE', 'UTC')

# 1. Localizar el único archivo CSV en la carpeta raw
# glob.glob devuelve una lista con los nombres de archivos que coinciden
archivos = glob.glob('../data/raw/*.csv')

# 2. Abrir el archivo (tomamos el primero de la lista [0])
if archivos:
    ruta_archivo = archivos[0]
    print(f"Archivo detectado y cargado: {ruta_archivo}")
    df = pd.read_csv(ruta_archivo)
    
    # Mostrar las primeras filas para confirmar
    display(df.head())
    display(df.info())

else:
    print("Error: No se encontró ningún archivo CSV en data/raw/")

ELIMNAMOS FILAS CON NULOS

In [ ]:
#Eliminamos filas con nulos.
df_clean = df.dropna()
df_clean.isnull().sum()

display (df_clean)
display (df_clean.info())





Redondeamos los campos numericos a dos digfitos.

In [ ]:
# Aplicamos redondeos
cols_refact = ['bmi', 'fatRate', 'bodyWaterRate', 'boneMass', 'muscleRate']
df_clean [cols_refact] = df_clean[cols_refact].round(2)

display (df_clean)
display (df_clean.info())

CAMBIAMOS LA COLUMNA TIME A DATETIME

In [ ]:
#Cambiamos a datetime y formateamos fecha
df_clean ['time'] = pd.to_datetime (df_clean['time'])
df_clean['time'] = df_clean['time'].dt.tz_convert(user_tz).dt.tz_localize(None)

#Ordenamos el dataframe por la fecha y reseteamos indices
df_clean = df_clean.sort_values(by='time', ascending=True)
df_clean = df_clean.reset_index(drop=True)

print(df_clean['time'].head())
display (df_clean)
display (df_clean.info())

Creamos nuevas columnas TIME

In [7]:
# 1. Extraer el componente de fecha (sin hora)
df_clean['date'] = df_clean['time'].dt.date

# 2. Extraer la hora (0-23)
df_clean['hour'] = df_clean['time'].dt.hour

# 3. Extraer el nombre del día de la semana (en inglés por defecto)
df_clean['day_name'] = df_clean['time'].dt.day_name()

# 4. Crear indicador de fin de semana (Sábado=5, Domingo=6)
# .dt.dayofweek devuelve 0 para Lunes y 6 para Domingo
df_clean['is_weekend'] = df_clean['time'].dt.dayofweek.isin([5, 6]).astype(int)

# 5. Visualizar las nuevas columnas
df_clean[['time', 'date', 'hour', 'day_name', 'is_weekend']].head()


display (df_clean)

,time,weight,height,bmi,fatRate,bodyWaterRate,boneMass,metabolism,muscleRate,visceralFat,date,hour,day_name,is_weekend
0,2023-03-28 22:30:18,100.10,192.0,27.10,29.09,50.63,3.62,1836.0,67.36,13.0,2023-03-28,22,Tuesday,0
1,2023-03-29 21:11:20,101.20,192.0,27.40,29.38,50.42,3.65,1852.0,67.82,13.0,2023-03-29,21,Wednesday,0
2,2023-03-30 21:25:15,100.80,192.0,27.30,29.29,50.49,3.64,1846.0,67.64,13.0,2023-03-30,21,Thursday,0
3,2023-04-01 21:00:20,99.85,192.0,27.00,29.15,50.59,3.61,1832.0,67.14,13.0,2023-04-01,21,Saturday,1
4,2023-04-03 21:19:26,100.10,192.0,27.10,28.95,50.73,3.63,1836.0,67.50,13.0,2023-04-03,21,Monday,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,2025-07-21 19:42:06,93.05,192.0,25.24,26.49,50.43,3.49,1713.0,64.91,13.0,2025-07-21,19,Monday,0
131,2025-07-24 12:49:16,93.10,192.0,25.25,26.51,50.41,3.49,1714.0,64.93,13.0,2025-07-24,12,Thursday,0
132,2025-07-30 12:21:50,92.65,192.0,25.13,26.31,50.55,3.48,1707.0,64.79,13.0,2025-07-30,12,Wednesday,0
133,2025-08-05 11:35:33,93.10,192.0,25.25,26.51,50.41,3.49,1714.0,64.93,13.0,2025-08-05,11,Tuesday,0


GUARDAMOS EL DATASET

In [13]:
# 1. Rutas exactas de los archivos
processed_path = '../data/processed/body_composition_master.csv'
garmin_pending_path = '../data/processed/garmin_pending.csv'

# --- FASE 1: DETECCIÓN DE DATOS REALMENTE NUEVOS ---

if os.path.exists(processed_path):
    # Cargar el histórico maestro tal y como está AHORA (antes de actualizarlo)
    df_master = pd.read_csv(processed_path)
    df_master['time'] = pd.to_datetime(df_master['time'])
    
    # Asegurarnos de que el nuevo dataframe también es datetime
    df_clean['time'] = pd.to_datetime(df_clean['time'])
    
    # LA CLAVE: Comparamos las fechas del archivo recién importado (df_clean) 
    # contra las del archivo maestro (df_master). 
    # Nos quedamos SÓLO con las filas cuya fecha NO está ya en el maestro.
    df_nuevos_reales = df_clean[~df_clean['time'].isin(df_master['time'])].copy()
    
else:
    # Si body_composition_master.csv no existe (primera vez que ejecutas en tu vida)
    df_master = pd.DataFrame(columns=df_clean.columns)
    df_nuevos_reales = df_clean.copy()

nuevos_registros = len(df_nuevos_reales)

# --- FASE 2: GUARDADO EN LOS DOS ARCHIVOS (Solo si hay datos nuevos) ---

if nuevos_registros > 0:
    print(f"✅ Se han detectado {nuevos_registros} pesajes nuevos.")
    
    # A. ACTUALIZAR body_composition_master.csv
    # Añadimos los datos nuevos al fondo del maestro
    df_final_master = pd.concat([df_master, df_nuevos_reales], ignore_index=True)
    df_final_master = df_final_master.sort_values(by='time').reset_index(drop=True)
    df_final_master.to_csv(processed_path, index=False)
    print(f"📊 body_composition_master.csv actualizado (Total: {len(df_final_master)} filas).")
    
    # B. ACTUALIZAR garmin_pending.csv
    if os.path.exists(garmin_pending_path):
        # Si existe, lo cargamos para añadirle los datos al final
        df_garmin = pd.read_csv(garmin_pending_path)
        df_garmin['time'] = pd.to_datetime(df_garmin['time'])
    else:
        # Si lo borraste tras exportar a FIT, creamos la estructura vacía
        df_garmin = pd.DataFrame(columns=df_clean.columns)
        
    df_garmin_updated = pd.concat([df_garmin, df_nuevos_reales], ignore_index=True)
    df_garmin_updated = df_garmin_updated.sort_values(by='time').reset_index(drop=True)
    df_garmin_updated.to_csv(garmin_pending_path, index=False)
    print(f"📥 garmin_pending.csv actualizado (Total pendientes: {len(df_garmin_updated)} filas).")

else:
    print("ℹ️ No hay datos nuevos. Ni body_composition_master.csv ni garmin_pending.csv han sido modificados.")

# --- FASE 3: LIMPIEZA DEL ARCHIVO DE ORIGEN ---

# Usamos la variable 'ruta_archivo' que definiste al principio para abrirlo
if os.path.exists(ruta_archivo):
    try:
        os.remove(ruta_archivo)
        print(f"🗑️  Archivo original eliminado de la carpeta raw: {os.path.basename(ruta_archivo)}")
    except Exception as e:
        print(f"⚠️  No se pudo eliminar el archivo original: {e}")
else:
    print("ℹ️  No se encontró el archivo original para eliminar (posiblemente ya fue borrado).")

ℹ️ No hay datos nuevos. Ni body_composition_master.csv ni garmin_pending.csv han sido modificados.
🗑️  Archivo original eliminado de la carpeta raw: BODY_1773136878051.csv
